<a href="https://colab.research.google.com/github/engMohamedAbdAlslam/DRP_segmentation/blob/copilot%2Fdevelop-preprocessing-pipeline/notebooks/01_disease_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Fundus Image Preprocessing (APTOS 2019)
**Dataset:** APTOS 2019 Blindness Detection — 3,662 fundus images, DR grading 0-4
**Goal:** Download via KaggleHub, preprocess fundus images (CLAHE + resize + normalize), save as `.npz`, visualize samples.
> Run on **Google Colab** with T4 GPU. Set your Kaggle credentials in Colab Secrets before running.

## 1. Colab Repo Setup

In [ ]:
import os
from pathlib import Path

repo_path = Path('/content/DRP_segmentation')
if not repo_path.exists():
    !git clone https://github.com/engMohamedAbdAlslam/DRP_segmentation.git /content/DRP_segmentation

%cd /content/DRP_segmentation
!git checkout copilot/develop-preprocessing-pipeline
!git pull origin copilot/develop-preprocessing-pipeline
print('Repo ready at', Path.cwd())

## 2. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 3. Kaggle Authentication

In [ ]:
from google.colab import userdata
import os

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
print('Kaggle credentials ready!')

## 4. Imports & Path Setup

In [ ]:
import sys, shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import sklearn
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

print('numpy', np.__version__)
print('opencv', cv2.__version__)

repo_root = Path('/content/DRP_segmentation')
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir      = repo_root / 'data'
raw_dir       = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Paths ready.')

## 5. Download APTOS 2019 Dataset

In [ ]:
import kagglehub

DATASET_SLUG = 'mariaherrerot/aptos2019'
DATASET_NAME = 'aptos2019'
aptos_root   = raw_dir / 'aptos2019'
aptos_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(aptos_root.rglob('*.png')) or any(aptos_root.rglob('*.jpg'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    shutil.copytree(str(download_path), str(aptos_root), dirs_exist_ok=True)
    print('Download complete:', aptos_root)
else:
    print('Dataset already present at:', aptos_root)

print(f'Total files: {len(list(aptos_root.rglob("*")))}') 

## 6. Index Images & Load DR Labels

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg'}
DR_GRADE_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative'}

# Load train labels CSV
csv_candidates = list(aptos_root.rglob('train.csv'))
if not csv_candidates:
    raise FileNotFoundError('train.csv not found. Check dataset download.')
label_csv = csv_candidates[0]
label_df  = pd.read_csv(label_csv)
print('CSV columns:', label_df.columns.tolist())
print(label_df.head())

# Find image folder
img_dirs = [p for p in aptos_root.rglob('*') if p.is_dir() and any(p.glob('*.png'))]
img_dir  = img_dirs[0] if img_dirs else aptos_root
print('Image folder:', img_dir)

# Build dataframe
id_col    = 'id_code' if 'id_code' in label_df.columns else label_df.columns[0]
grade_col = 'diagnosis' if 'diagnosis' in label_df.columns else label_df.columns[1]

rows = []
for _, row in label_df.iterrows():
    img_path = img_dir / (str(row[id_col]) + '.png')
    if not img_path.exists():
        img_path = img_dir / (str(row[id_col]) + '.jpg')
    if img_path.exists():
        rows.append({'image_path': img_path, 'dr_grade': int(row[grade_col])})

df = pd.DataFrame(rows)
print(f'\nTotal matched images: {len(df)}')
print('\nDR Grade distribution:')
print(df['dr_grade'].map(DR_GRADE_NAMES).value_counts().sort_index())

## 7. Train / Val / Test Split (70 / 15 / 15) — Stratified

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check Cell 6.')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['dr_grade'])
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['dr_grade'])

splits = {'train': train_df, 'val': val_df, 'test': test_df}
for name, sdf in splits.items():
    print(f'{name}: {len(sdf)} images | grade dist: {sdf["dr_grade"].value_counts().sort_index().to_dict()}')

## 8. Batch Preprocessing & Save as .npz

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            result   = preprocess_fundus_image(row['image_path'], mask=None, config=config)
            out_file = out_dir / (row['image_path'].stem + '.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})

print(f'\nDone. Errors: {len(errors)}')
if errors:
    print(pd.DataFrame(errors))

## 9. Dataset Statistics

In [ ]:
print('=== Processed .npz Counts ===')
for split_name in splits:
    count = len(list((processed_dir / DATASET_NAME / split_name).rglob('*.npz')))
    print(f'  {split_name}: {count} files')

# Image size sample
sample_imgs = list(aptos_root.rglob('*.png'))[:200]
widths, heights = [], []
for p in sample_imgs:
    img = cv2.imread(str(p))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w); heights.append(h)

if widths:
    print(f'\nOriginal size sample ({len(widths)} imgs):')
    print(f'  Width  — min:{min(widths)} max:{max(widths)} mean:{int(np.mean(widths))}')
    print(f'  Height — min:{min(heights)} max:{max(heights)} mean:{int(np.mean(heights))}')

## 10. Visualization — Sample Grid by DR Grade

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for grade in range(5):
    subset = train_df[train_df['dr_grade'] == grade]
    if subset.empty:
        axes[grade].axis('off')
        continue
    sample = subset.sample(1, random_state=42).iloc[0]
    img_bgr = cv2.imread(str(sample['image_path']))
    if img_bgr is None:
        axes[grade].axis('off')
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[grade].imshow(img_rgb)
    axes[grade].set_title(f'Grade {grade}\n{DR_GRADE_NAMES[grade]}', fontsize=10, fontweight='bold')
    axes[grade].axis('off')

plt.suptitle('APTOS 2019 — One Sample per DR Grade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Processed Files to Google Drive
> Run once after Cell 8 to persist `.npz` files across sessions.
> Saved to: `My Drive/DRP_processed/aptos2019/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_dest = Path('/content/drive/MyDrive/DRP_processed')
drive_dest.mkdir(parents=True, exist_ok=True)

src = processed_dir / DATASET_NAME
if src.exists():
    shutil.copytree(str(src), str(drive_dest / DATASET_NAME), dirs_exist_ok=True)
    npz_count = len(list((drive_dest / DATASET_NAME).rglob('*.npz')))
    print(f'Saved {npz_count} .npz files → {drive_dest / DATASET_NAME}')
else:
    print('No processed files found. Run Cell 8 first.')

## Next Steps — Notebook 02
- Load `.npz` from `data/processed/aptos2019/` (or Google Drive)
- Build a **DR grading classifier**: EfficientNet-B4 or ResNet-50
- Loss: cross-entropy with class weights (handle grade imbalance)
- Metrics: **Quadratic Weighted Kappa**, **AUC-ROC**, **Accuracy per grade**
- Explainability: **Grad-CAM** on predicted grade to highlight retinal regions